In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.inspection import permutation_importance
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

In [2]:
model = joblib.load('../app/data/hospital_closure_model.pkl')
test = pd.read_csv('../data/test_data_with_pred.csv')
train = pd.read_csv('../data/train_set.csv')
hospitals_full = pd.read_csv('../data/hospitals_full.csv')
hospitals_master = pd.read_csv('../data/hospitals_master.csv')
surv_functions = pd.read_csv('../data/surv_functions.csv')

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_81411/2087134432.py:4: DtypeWarning: Columns (67) have mixed types. Specify dtype option on import or set low_memory=False.
  hospitals_full = pd.read_csv('../data/hospitals_full.csv')


In [3]:
test['CCN'] = test['CCN'].astype(int).astype(str).str.zfill(6)

In [4]:
# Merge hospitals master with test data to bring back in identifying characteristics like hospital name
test_full = hospitals_master[['CCN','Facility Name','Prior Name','State','Closure Date']].merge(test, how='right', on='CCN')

In [5]:
test_full.head(2)

,CCN,Facility Name,Prior Name,State,Closure Date,Medicaid charges,STATEMENT OF REVENUES AND EXPENSES: Net Income (G3_C1_29),"ADJUSTED SALARIES, Subtotal Salaries",BALANCE SHEET: Prepaid expenses (G_C1THRU4_8),BED DAYS: Total Hospital,...,Hospital Type_Voluntary non-profit - Church,Hospital Type_Voluntary non-profit - Other,Hospital Type_Voluntary non-profit - Private,General Ownership Type_for-profit,General Ownership Type_govt,General Ownership Type_non-profit,Pred_Closure,Status,Time,Year
0,190044,ACADIA GENERAL HOSPITAL,NaN,LA,NaN,-0.141395,-0.291649,-0.496079,-0.494961,-0.456705,...,0.0,0.0,1.0,0.0,0.0,1.0,0.311765,False,16.0,2022
1,190044,ACADIA GENERAL HOSPITAL,NaN,LA,NaN,-0.244175,0.035763,-0.464594,-0.445257,-0.456705,...,0.0,0.0,1.0,0.0,0.0,1.0,0.145122,False,16.0,2023


Looking at survival function dataframe. Each value in the survival functions dataframe captures the probability that the given hospital will survive (remain open) beyond the year in the column.

In [6]:
surv_functions.columns = [2009 + int(float(col)) for col in surv_functions.columns]

In [7]:
surv_functions_full = pd.concat([test_full[['CCN','Facility Name','Year']], surv_functions], axis=1)
surv_functions_full.head(2)

,CCN,Facility Name,Year,2011,2012,2014,2015,2016,2020,2021,2022,2023,2024,2025,2026
0,190044,ACADIA GENERAL HOSPITAL,2022,0.999901,0.999802,0.999703,0.999603,0.999504,0.999105,0.998903,0.998903,0.998903,0.998903,0.998903,0.998903
1,190044,ACADIA GENERAL HOSPITAL,2023,0.999916,0.999832,0.999748,0.999664,0.999580,0.999243,0.999071,0.999071,0.999071,0.999071,0.999071,0.999071


In [8]:
surv_functions_full[surv_functions_full['Year']==2025][['Facility Name',2025]].sort_values(by=2025)

,Facility Name,2025
465,CARILION TAZEWELL COMMUNITY HOSPITAL,0.913160
1600,MARION REGIONAL MEDICAL CENTER,0.988114
940,FAYETTE MEDICAL CENTER,0.995643
2959,UNION MEDICAL CENTER,0.996196
1101,GREENE COUNTY HOSPITAL,0.996591
...,...,...
1961,NANTUCKET COTTAGE HOSPITAL,0.999306
1295,HUGH CHATHAM MEMORIAL HOSPITAL,0.999308
1686,MCLEOD MEDICAL CENTER - DILLON,0.999310
1065,GRADY GENERAL HOSPITAL,0.999338


Extracting feature coefficients to see exactly how much each feature increases or decreases the log-hazard risk. A positive coefficient indicates a higher risk of hospital closure, while a negative coefficient indicates a protective effect.

In [9]:
feature_cols = [col for col in test.columns if col not in ('CCN','Pred_Closure','Status','Time','Year')]

feature_coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[:, 20] 
})

# Filter to see only the non-zero selected features
selected_features = feature_coef_df[feature_coef_df['Coefficient'] != 0]

In [10]:
pd.set_option('display.max_colwidth', None)
selected_features.sort_values(by='Coefficient').reset_index()

,index,Feature,Coefficient
0,77,Per Capita Short Term Gen Hosp Admissions,-0.866187
1,114,Hospital Type_Voluntary non-profit - Private,-0.752585
2,66,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals,-0.252524
3,81,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,-0.188339
4,47,Financial Indicators: SOLVENCY Equity Ratio,-0.160108
5,79,Per Capita Total Active D.O.s Non-Federal,0.150631
6,46,Financial Indicators: SOLVENCY Debt Ratio,0.167392
7,113,Hospital Type_Voluntary non-profit - Other,0.395019
8,117,General Ownership Type_non-profit,0.597898


Looking at the most impactful model features based on permutation importance scores. 

To evaluate the model performance, I'll use concordance_index_censored, which measures how accurately the model ranks hospitals by their risk of closure over time.

Permutation feature importance will measure how much the model's concordance_index_censored score drops when the values of a single feature are randomly shuffled.

In [11]:
feature_cols = [col for col in test.columns if col not in ('CCN','Closure Proximity','Status','Time','Pred_Closure','Year')]
X_test = test[feature_cols]
X_train = train[feature_cols]
y_test = test[['Status','Time']]

def c_index_scorer(model, X, y):
    predictions = model.predict(X, alpha=model.alphas_[20])
    
    event_indicator = y_test['Status']
    event_time = y_test['Time']
    
    c_index, _, _, _, _ = concordance_index_censored(event_indicator, event_time, predictions)
    
    return c_index

# Permutation importance applied to the CoxnetSurvivalAnalysis model
feature_importances = permutation_importance(
    model, 
    X_test.to_numpy(), 
    Surv.from_arrays(y_test['Status'], y_test['Time']),
    random_state=0, 
    scoring=c_index_scorer,
    n_repeats=5 
)

# Create the results DataFrame
top_features_df = pd.DataFrame({
    'variable': feature_cols,
    'importance': feature_importances['importances_mean']
}).sort_values('importance', ascending=False)

In [12]:
top_features_df.head(10)

,variable,importance
114,Hospital Type_Voluntary non-profit - Private,0.134668
117,General Ownership Type_non-profit,0.063682
77,Per Capita Short Term Gen Hosp Admissions,0.037844
66,Dist Hosp By 00 - 39% Util Rate Short Term General Hospitals,0.011270
81,Per Capita Total Medicare Inpatient Days Short Term General Hospitals,0.010845
113,Hospital Type_Voluntary non-profit - Other,0.004927
46,Financial Indicators: SOLVENCY Debt Ratio,0.003060
47,Financial Indicators: SOLVENCY Equity Ratio,0.002683
79,Per Capita Total Active D.O.s Non-Federal,0.000929
65,% <65 without Health Insurance,0.000000


In [13]:
top_features_df.tail(10)

,variable,importance
39,Financial Indicators: Operating Profit,0.0
38,Financial Indicators: Net Profit Margin,0.0
37,Financial Indicators: Cash Reserves,0.0
36,S-10 DATA: Medicaid Costs,0.0
35,Financial Indicators: Total Net Assets,0.0
34,STATEMENT OF REVENUES AND EXPENSES: Net Patient Revenue (G3_C1_3),0.0
33,BALANCE SHEET: Inventory (G_C1THRU4_7),0.0
32,BALANCE SHEET: Total Current Liabilities (G_C1THRU4_45),0.0
31,"RECONCILIATION OF CAPITAL COST CENTERS: Depreciation, Total (A7_3_C9_3)",0.0
59,info_score,0.0


In [14]:
# Fixed time point will be 2025 (final year in the dataset)
time_horizon = 16.0 # years since 2010 (first year in the dataset), including 2010

def predict_survival_prob(X_input):
    # Get the list of survival functions for all samples
    surv_funcs = model.predict_survival_function(X_input, alpha=model.alphas_[20])
    
    # Evaluate each step function at the specific year of interest (2025)
    # This gives us a 1D array of probabilities between 0 and 1
    probabilities = np.array([func(time_horizon) for func in surv_funcs])
    return probabilities


In [ ]:
# Sample the train data for faster runtime
X_train_summary = shap.sample(X_train.to_numpy(), nsamples=50)

# Pass the sampled train data to the explainer
explainer = shap.KernelExplainer(predict_survival_prob, X_train_summary)

# Compute SHAP values for test data
shap_values = explainer.shap_values(X_test.to_numpy())

  0%|          | 0/3310 [00:00<?, ?it/s]

In [ ]:
shap_values.feature_names = feature_cols

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.bar(shap_values)

Case Studies:

Hospitals with the highest chance of closing, per our model:

In [ ]:
test_full[['Facility Name','Pred_Closure','Year','Status','Time']].sort_values(by='Pred_Closure',ascending=False).head(5)

Identifying good case studies based on facilities with the highest variance in their risk scores:

In [ ]:
test_full.groupby(['CCN','Facility Name'])['Pred_Closure'].var().nlargest(5)

In [ ]:
test_full[test_full['Facility Name']=='HAZARD ARH REGIONAL MEDICAL CENTER'][['Facility Name','Year','Pred_Closure']].sort_values(by='Year')

#### SHAP plots:

In [ ]:
shap.plots.waterfall(shap_values[1186])

In [ ]:
shap.plots.waterfall(shap_values[1187])

In [ ]:
shap.plots.waterfall(shap_values[1188])

In [ ]:
shap.plots.waterfall(shap_values[1189])

#### 

In [ ]:
test_full[test_full['Facility Name']=='CHELSEA HOSPITAL'][['Facility Name','Year','Pred_Closure']].sort_values(by='Year')

In [ ]:
test_full[test_full['Facility Name']=='COVENANT HEALTH HOBBS HOSPITAL'][['Facility Name','Year','Pred_Closure']].sort_values(by='Year')

Finding a hospital that closed that could be a good case study:

In [ ]:
test_full[test_full['Status']==True].groupby(['CCN','Facility Name'])['Pred_Closure'].agg(lambda x: x.max() - x.min()).nlargest(5)

In [ ]:
test_full[test_full['Status']==True][['Facility Name','Year','Pred_Closure']].sort_values(by='Pred_Closure',ascending=False).head(5)

In [ ]:
test_full[test_full['Facility Name']=='GLENN MEDICAL CENTER'][['CCN','Facility Name','Year','Pred_Closure']].sort_values(by='Year')

In [ ]:
# COVENANT HEALTH HOBBS HOSPITAL in 2024
shap.plots.waterfall(shap_values[759])

In [ ]:
# CARILION TAZEWELL COMMUNITY HOSPITAL in 2025
shap.plots.waterfall(shap_values[465])

In [ ]:
# MARION REGIONAL MEDICAL CENTER in 2025
shap.plots.waterfall(shap_values[1600])